In [1]:
import  pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
from collections import Counter

In [2]:
df_cbf = pd.read_csv('data/cbf_df.csv')
df_traing = pd.read_csv('data_spliting2/training.csv')

In [3]:
lista_compilado = df_cbf["geral"].tolist()

print(lista_compilado)

['Adventure Animation Children Comedy Fantasy pixar pixar fun 1995', 'Adventure Children Fantasy fantasy magic board game Robin Williams game 1995', 'Comedy Romance moldy old 1995', 'Comedy Drama Romance  1995', 'Comedy pregnancy remake 1995', 'Action Crime Thriller  1995', 'Comedy Romance remake 1995', 'Adventure Children  1995', 'Action  1995', 'Action Adventure Thriller  1995', 'Comedy Drama Romance politics president 1995', 'Comedy Horror  1995', 'Adventure Animation Children  1995', 'Drama politics president 1995', 'Action Adventure Romance  1995', 'Crime Drama Mafia 1995', 'Drama Romance Jane Austen 1995', 'Comedy  1995', 'Comedy  1995', 'Action Comedy Crime Drama Thriller  1995', 'Comedy Crime Thriller Hollywood 1995', 'Crime Drama Horror Mystery Thriller serial killer 1995', 'Action Crime Thriller  1995', 'Drama Sci-Fi  1995', 'Drama Romance alcoholism 1995', 'Drama Shakespeare 1995', 'Children Drama  1995', 'Drama Romance In Netflix queue Jane Austen 1995', 'Adventure Drama Fa

In [4]:
cvec = CountVectorizer(stop_words='english', min_df=3, max_df=0.5, ngram_range=(1,2))

sf = cvec.fit_transform(lista_compilado)

###matriz de vocabulario do documento
print(sf.toarray())

### Vocabulários obtidos após o aprendizado (função Fit)
cvec.vocabulary_

[[0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 ...
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]]


{'adventure': 134,
 'animation': 170,
 'children': 276,
 'comedy': 313,
 'fantasy': 646,
 'pixar': 1090,
 'fun': 778,
 '1995': 76,
 'adventure animation': 139,
 'animation children': 178,
 'children comedy': 285,
 'comedy fantasy': 377,
 'magic': 949,
 'robin': 1143,
 'williams': 1521,
 'adventure children': 140,
 'children fantasy': 288,
 'fantasy fantasy': 687,
 'fantasy magic': 690,
 'robin williams': 1144,
 'romance': 1149,
 'comedy romance': 386,
 'drama': 512,
 'comedy drama': 376,
 'drama romance': 618,
 'romance 1995': 1195,
 'pregnancy': 1102,
 'remake': 1138,
 'action': 104,
 'crime': 402,
 'thriller': 1325,
 'action crime': 119,
 'crime thriller': 442,
 'thriller 1995': 1362,
 'children 1995': 279,
 'action 1995': 110,
 'action adventure': 115,
 'adventure thriller': 150,
 'politics': 1095,
 'president': 1103,
 'politics president': 1096,
 'horror': 828,
 'comedy horror': 380,
 'horror 1995': 849,
 'drama politics': 615,
 'adventure romance': 148,
 'mafia': 947,
 'crime dram

In [5]:
transformer = TfidfTransformer()
tfidf_matrix = transformer.fit_transform(sf)
print(tfidf_matrix)

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 58300 stored elements and shape (9708, 1530)>
  Coords	Values
  (0, 76)	0.20323780538036668
  (0, 134)	0.13338205906471118
  (0, 139)	0.2047971262360291
  (0, 170)	0.16530414159925821
  (0, 178)	0.19617724730554828
  (0, 276)	0.1612622274449093
  (0, 285)	0.18824289588966098
  (0, 313)	0.08554170058012703
  (0, 377)	0.2071519638685455
  (0, 646)	0.15465174240174193
  (0, 778)	0.3616459154348297
  (0, 1090)	0.7528450333065222
  (1, 76)	0.19047172554846023
  (1, 134)	0.12500386382206238
  (1, 140)	0.21170939788470713
  (1, 276)	0.15113278097908145
  (1, 288)	0.2298942278161104
  (1, 646)	0.28987504740278364
  (1, 687)	0.352778097303578
  (1, 690)	0.3619621777276919
  (1, 949)	0.3334338182276389
  (1, 1143)	0.3619621777276919
  (1, 1144)	0.3619621777276919
  (1, 1521)	0.3389296662670953
  (2, 76)	0.65423335183974
  :	:
  (9703, 104)	0.1806806017089365
  (9703, 116)	0.4037611516395078
  (9703, 170)	0.25494230388020334
  (9703, 1

In [6]:
cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)
cosine_sim.mean()

np.float64(0.03310770075912247)

In [7]:
df_ratings_count = pd.read_csv('data/movies_ratings_count.csv')

list_ratings_count = df_ratings_count.loc[:, 'count_ratings'].to_list()

list_ratings_count = np.array(list_ratings_count)

list_ratings_count.shape

log_ratings = np.log1p(list_ratings_count)
log_ratings


# 2. Normalização Min-Max (coloca tudo entre 0 e 1)
# Subtraímos o mínimo e dividimos pela amplitude (max - min)
pop_score = (log_ratings - log_ratings.min()) / (log_ratings.max() - log_ratings.min())
pop_score.shape
pop_score.mean()

np.float64(0.2848063003983221)

In [8]:
def recomendar_filmes(titulo_filme, movies_df, cosine_sim_matrix, n_recomendacoes=10):
    print(f'gerando recomendações para filme {titulo_filme}\n')
    indices = pd.Series(movies_df.index, index=movies_df['titulo_movie_lens']).drop_duplicates()
    
    if titulo_filme not in indices:
        print(f"Filme '{titulo_filme}' não encontrado no dataset.")
        return pd.Series([])

    idx = indices[titulo_filme]

    sim_scores = list(enumerate(cosine_sim_matrix[idx]))



    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    sim_scores = sim_scores[1:n_recomendacoes + 1]  

    filme_indices = [i[0] for i in sim_scores]
    print(f"filmes similares ao {titulo_filme}")
    print(movies_df['titulo_movie_lens'].iloc[filme_indices].to_string(index=False))
    return movies_df['titulo_movie_lens'].iloc[filme_indices]



filme = "Toy Story (1995)"
recomendados = recomendar_filmes(filme, df_cbf, cosine_sim)





gerando recomendações para filme Toy Story (1995)

filmes similares ao Toy Story (1995)
                              Bug's Life, A (1998)
                                Toy Story 2 (1999)
Twelve Tasks of Asterix, The (Les douze travaux...
                                         Up (2009)
                    Tale of Despereaux, The (2008)
                                  Wild, The (2006)
Asterix and the Vikings (Astérix et les Vikings...
                          The Good Dinosaur (2015)
                            Shrek the Third (2007)
                                      Moana (2016)


In [9]:


def recomendar_filmes_perfil(user_id, movies_df, cosine_sim_matrix, pop_score, n_recomendacoes=50, df_traing_path='data_spliting/traing.csv'):

    # 1. Carregar o histórico de treino do usuário
    df_traing = pd.read_csv(df_traing_path) 

    # 2. Identificar os IDs dos filmes que o usuário já assistiu

    df_user = df_traing.loc[df_traing['userId'] == user_id]

    list_id_movies = df_user['movieId'].to_list()

    # 3. Mapear esses IDs para os índices numéricos

    indices_usuario = movies_df[movies_df['movieId'].isin(list_id_movies)].index.tolist()

    if not indices_usuario:

        return pd.DataFrame()

    # 4. Cálculo da Recomendação por Perfil (Similaridade Média)

    # Pegamos a média das linhas da matriz de similaridade para os filmes assistidos

    sim_scores_total = np.mean(cosine_sim_matrix[indices_usuario], axis=0)

    # --- NOVIDADE: APLICANDO O POPULARITY BIAS ---
    # Normalizamos a similaridade para 0-1 (garante que os pesos funcionem bem)
    # Se sim_scores_total for todo zero (raro), evitamos divisão por zero
    max_sim = sim_scores_total.max()
    if max_sim > 0:

        sim_scores_total = sim_scores_total / max_sim

    # Definimos os pesos (Ajuste aqui: mais conteúdo ou mais popularidade)

    peso_conteudo = 0.8
    peso_pop = 0.2

    # Calculamos o score híbrido
    # Como ambos são arrays de mesmo tamanho (9708,), a conta é direta
    score_hibrido = (sim_scores_total * peso_conteudo) + (pop_score * peso_pop)
    # Criamos a lista de (índice, score) usando o score híbrido
    sim_scores = list(enumerate(score_hibrido))
    # 5. Ordenação e Filtragem
    # Ordenamos pelos scores híbridos mais altos
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)

    # Remover filmes que o usuário já assistiu
    sim_scores_filtrados = [s for s in sim_scores if s[0] not in indices_usuario]

    # 6. Selecionar o Top N
    top_sim_scores = sim_scores_filtrados[:n_recomendacoes]
    filme_indices = [i[0] for i in top_sim_scores]
    # 7. Gerar o resultado
    recomendacoes = movies_df.iloc[filme_indices][['movieId', 'titulo_movie_lens']].copy()
    recomendacoes['score_recomendacao'] = [i[1] for i in top_sim_scores]

    return recomendacoes

    

recomendados = recomendar_filmes_perfil(3, df_cbf, cosine_sim, pop_score, 100)
recomendados

,movieId,titulo_movie_lens,score_recomendacao
3746,5219,Resident Evil (2002),0.868286
2043,2722,Deep Blue Sea (1999),0.856968
1009,1320,Alien³ (a.k.a. Alien 3) (1992),0.856513
5312,8810,AVP: Alien vs. Predator (2004),0.845815
5328,8861,Resident Evil: Apocalypse (2004),0.839527
...,...,...,...
2724,3658,Quatermass and the Pit (1967),0.681040
522,610,Heavy Metal (1981),0.680666
4056,5779,Piranha II: The Spawning (1981),0.680144
2822,3770,Dreamscape (1984),0.680082


In [15]:
def preparar_generos_binarios(df_alvo, df_cbf):
    """
    df_alvo: O dataframe que você quer expandir (ex: recomendações ou treino)
    df_cbf: O dataframe original com a coluna 'genres'
    """
    # 1. Isolar os gêneros e criar dummies
    # Usamos o df_cbf para garantir que todos os gêneros possíveis do dataset estejam presentes
    dummies = df_cbf['genres'].str.get_dummies(sep='|')
    movies_genres = pd.concat([df_cbf[['movieId']], dummies], axis=1)
    
    # 2. Fazer o merge com o dataframe alvo
    df_expandido = df_alvo.merge(movies_genres, on='movieId', how='left')
    
    # Retorna a lista de nomes das colunas de gêneros (para facilitar o uso depois)
    colunas_generos = dummies.columns.tolist()
    
    return df_expandido, colunas_generos



def validar_generos_recomendados(user_id, recomendacoes, training_genres, df_cbf, k=10):
    """
    Valida as recomendações baseando-se nos gêneros já normalizados no treino.
    """
    # 1. Identificar as colunas de gêneros (excluindo IDs e metadados)
    # Pegamos as colunas do training_genres que não são as básicas
    cols_nao_genero = ['userId', 'movieId', 'rating', 'timestamp', 'genres', 'titulo_movie_lens']
    lista_generos = [c for c in training_genres.columns if c not in cols_nao_genero]

    # 2. Obter o Perfil do Usuário (Baseado no seu df de treino já pronto)
    user_train = training_genres[training_genres['userId'] == user_id]
    if user_train.empty:
        return 0.0, 0.0

    # Gêneros que o usuário assistiu (Top 3)
    perfil_usuario = user_train[lista_generos].sum().sort_values(ascending=False)
    generos_favoritos = perfil_usuario[perfil_usuario > 0].head(3).index.tolist()

    print(generos_favoritos)

    if not generos_favoritos:
        return 0.0, 0.0

    # 3. Preparar as Recomendações (Anexar os gêneros do df_cbf)
    # Criamos os dummies apenas para o df_cbf para buscar as infos
    dummies_cbf = df_cbf['genres'].str.get_dummies(sep='|')
    movies_generos_info = pd.concat([df_cbf[['movieId']], dummies_cbf], axis=1)
    
    # Fazemos o merge com o top K das recomendações
    rec_top_k = recomendacoes.head(k).merge(movies_generos_info, on='movieId', how='left')


    hits = 0
    for _, filme in rec_top_k.iterrows():
        if any(filme[gen] == 1 for gen in generos_favoritos):
            hits += 1
    
    precision_at_k = hits / k


    generos_entregues = 0
    for gen in generos_favoritos:
        if gen in rec_top_k.columns and rec_top_k[gen].sum() > 0:
            generos_entregues += 1
            
    recall_at_k = generos_entregues / len(generos_favoritos)

    return precision_at_k, recall_at_k

recomendados = recomendar_filmes_perfil(6, df_cbf, cosine_sim, pop_score, 100)
validar_generos_recomendados(6, recomendados, df_traing, df_cbf, 100)


['Drama', 'Comedy', 'Thriller']


(0.98, 1.0)

In [12]:
def validar_modelo_por_perfil_treino(df_treino_normalizado, df_cbf, cosine_sim_matrix, pop_score, k=10):
    """
    Avalia o modelo comparando as recomendações com o perfil histórico (treino) do usuário.
    """
    # 1. Preparar metadados dos filmes (dummies) apenas uma vez
    dummies_cbf = df_cbf['genres'].str.get_dummies(sep='|')
    movies_generos_info = pd.concat([df_cbf[['movieId']], dummies_cbf], axis=1)
    
    # Identificar quais colunas no df_treino são os gêneros (Action, Comedy, etc)
    cols_nao_genero = ['userId', 'movieId', 'rating', 'timestamp', 'genres', 'titulo_movie_lens']
    lista_generos = [c for c in df_treino_normalizado.columns if c not in cols_nao_genero]

    # Lista de usuários que possuem histórico no treino
    usuarios = df_treino_normalizado['userId'].unique()
    
    total_precision = 0
    total_recall = 0
    contagem_usuarios = 0

    for user_id in usuarios:
        # 2. Gerar recomendações para o usuário (Motor Híbrido)
        rec_df = recomendar_filmes_perfil(user_id, df_cbf, cosine_sim_matrix, pop_score, n_recomendacoes=k)
        
        if rec_df.empty:
            continue

        # 3. Extrair o Gosto do Usuário do treino
        user_history = df_treino_normalizado[df_treino_normalizado['userId'] == user_id]
        # Somamos a ocorrência de cada gênero e pegamos os 3 principais
        perfil_usuario = user_history[lista_generos].sum().sort_values(ascending=False)
        generos_favoritos = perfil_usuario[perfil_usuario > 0].head(3).index.tolist()

        if not generos_favoritos:
            continue

        # 4. Cruzar Recomendações com Gêneros
        rec_com_info = rec_df.merge(movies_generos_info, on='movieId', how='left')

        # 5. Cálculo de Precision@K (Hits de Gênero)
        hits = 0
        for _, filme in rec_com_info.iterrows():
            if any(filme[gen] == 1 for gen in generos_favoritos):
                hits += 1
        
        precision_user = hits / k

        # 6. Cálculo de Recall@K (Cobertura dos gêneros favoritos)
        generos_entregues = 0
        for gen in generos_favoritos:
            if gen in rec_com_info.columns and rec_com_info[gen].sum() > 0:
                generos_entregues += 1
        
        recall_user = generos_entregues / len(generos_favoritos)

        total_precision += precision_user
        total_recall += recall_user
        contagem_usuarios += 1

    return {
        'Mean Genre Precision@K': total_precision / contagem_usuarios,
        'Mean Genre Recall@K': total_recall / contagem_usuarios,
        'Total Usuários': contagem_usuarios
    }


validar_modelo_por_perfil_treino(df_traing, df_cbf, cosine_sim, pop_score, 100)

{'Mean Genre Precision@K': 0.9600986842105277,
 'Mean Genre Recall@K': 1.0,
 'Total Usuários': 608}

In [ ]:
numeros = list(range(1, 600))
def verificar_genericidade(lista_usuarios, pasta='filmes_recomendados_users'):
        todas_recomendações = []
        
        for uid in lista_usuarios:
            try:
                df = pd.read_csv(f'{pasta}/user{uid}.csv')
                todas_recomendações.extend(df['movieId'].tolist())
            except Exception as e:
                print(f'usuaário {uid}não tem csv e filmes recomenandos pois não avaliou filmes')

        
        # Conta quais filmes mais apareceram entre todos os usuários testados
        contagem = Counter(todas_recomendações)
        most_common = contagem.most_common(10)
        
        print("Filmes que mais apareceram em todas as recomendações:")
        for movie_id, freq in most_common:
            # Aqui você pode buscar o título no movies_df para facilitar a leitura
            print(f"Movie ID: {movie_id} | Apareceu para {freq} usuários")
# Teste com uma amostra de 50 usuários
verificar_genericidade(numeros[:598])

usuaário 214não tem csv e filmes recomenandos pois não avaliou filmes
usuaário 442não tem csv e filmes recomenandos pois não avaliou filmes
Filmes que mais apareceram em todas as recomendações:
Movie ID: 1544 | Apareceu para 298 usuários
Movie ID: 4638 | Apareceu para 285 usuários
Movie ID: 1356 | Apareceu para 280 usuários
Movie ID: 2628 | Apareceu para 260 usuários
Movie ID: 8361 | Apareceu para 256 usuários
Movie ID: 849 | Apareceu para 253 usuários
Movie ID: 480 | Apareceu para 249 usuários
Movie ID: 91500 | Apareceu para 248 usuários
Movie ID: 48774 | Apareceu para 244 usuários
Movie ID: 380 | Apareceu para 239 usuários
